# Applying the XGBoost Algorithm

This is the worked solution. XGBoost is a gradient-boosting library that builds an additive ensemble of shallow trees, where each new tree corrects the errors of the ones before it. Here we apply it to the Pima Native Americans diabetes dataset: prepare the data, train an `XGBClassifier`, evaluate it, then map and visualise the decision surface it has learned.

The dataset ships without column names, so we add them after loading.

## Learning Objectives

At the end of this notebook, you should be able to:

- Train an XGBoost classifier on a real medical dataset.
- Evaluate a classifier with a classification report and a confusion matrix.
- Rank feature importances to see which inputs drive the predictions.
- Map and visualise a model's decision surface over a grid of feature values.

## Import and Setup

In [ ]:
# Import modules
from xgboost import XGBClassifier

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from itertools import product

import warnings
warnings.filterwarnings("ignore")

RSEED=42

## Load and Prepare the Data

In [ ]:
# Import data
df = pd.read_csv('../data/pima-native-americans-diabetes.csv', header=None)
column_names = ['pregnancies', 'glucose', 'blood_pressure', 'skin_thickness', 'insulin', 'bmi', 'diabetes_pedigree_function', 'age', 'outcome']
# Set columns names of data frame
df.columns = column_names

In [ ]:
df.head()

## Train and Evaluate the Model

XGBoost needs little preparation: it does not require feature scaling and handles the raw ranges as they are. We hold out 30% of the patients for testing so the scores reflect performance on data the model has not seen.

In [ ]:
# Split the data
X = df.drop('outcome', axis=1)
y = df['outcome']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=RSEED)

In [ ]:
# Fit model to training data
xgb = XGBClassifier(random_state=RSEED,
                    n_jobs=-1,
                    #n_estimators=1000,
                    #learning_rate=0.3,
                    #subsample=0.5,
                    )
xgb.fit(X_train, y_train)
# Make predictions on test set
y_pred = xgb.predict(X_test)

In [ ]:
# Evaluate your model
print(classification_report(y_test, y_pred))

The model reaches about **0.71** accuracy on the held-out patients. More informative on this imbalanced set is the positive class (diabetes): precision around **0.57** and recall around **0.69**, so the model catches roughly two thirds of the diabetic patients but raises a fair number of false alarms. Accuracy alone would mask this, which is why we read the per-class precision and recall.

In [ ]:
# Show confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot with seaborn
sns.heatmap(cm, annot=True, fmt='g', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')

The matrix shows about **110** true negatives and **55** true positives, against **41** false positives and **25** false negatives. The 25 false negatives are the costly errors here: diabetic patients the model labelled as healthy. On a medical screen you would usually accept some extra false positives to push recall up and shrink that number.

## Mapping the Decision Surface

Instead of predicting a few individual values, a useful trick is to predict for a whole matrix of feature combinations rather than just the test set. Evaluating the model across a regular grid lets us read off its **decision surface**: how the predicted probability changes as the inputs change. It also produces a fast lookup table that approximates the model.

In [ ]:
#Function to produce a row with regular intervals
def make_row(row_length,row_min,row_max,):
    step = (row_max - row_min)/row_length
    row = []
    for i in range(0,row_length+1):
#        print(df._STATE.min()+(step*i))
        row.append(row_min+(step*i))

    return row



In [ ]:
#Produce a grid. 
grid = pd.DataFrame()

for i in X_train.columns:
    row_min = X_train[i].min()
    row_max = X_train[i].max()
    row = make_row(3,row_min,row_max) #Selecting a high value here will result in long waiting time in the next step
    grid[i] = row

grid

In [ ]:
#Create permutations of all possible values

def expand_grid(dictionary):
   return pd.DataFrame([row for row in product(*dictionary.values())], 
                       columns=dictionary.keys())

dictionary = {}
for i in grid.columns:
    dictionary.update({i : grid[i]})

full_grid = expand_grid(dictionary)

full_grid


In [ ]:
#predict the full grid using the model
prob = xgb.predict_proba(full_grid)

#extract positive predictions and assign them to a column in full_grid
full_grid['prob'] = prob[:,1]



In [ ]:
full_grid.tail(100)

Above you can see the last 100 rows from the matrix we have created with the probability of diabetes calculated. This lookup table is an approximation of the model we've created - although it is not as precise as the model it can be used for various purposes - as plan B for your main model when it goes offline or as way to visualise the representation of the world your model has built. 

In [ ]:
#save the grid to a csv file - be ware of exceptionally large csv files
full_grid.to_csv('../data/full_grid.csv')



## Feature Importance

XGBoost records how much each feature contributed to reducing impurity across all the trees. Ranking these shows which measurements the model leans on most.

In [ ]:
#extract the most important features from the model
imp = list(zip(xgb.feature_names_in_,xgb.feature_importances_))
imp.sort(key = lambda tuple:tuple[1],reverse= True)
imp[0:2]

**Glucose** dominates (importance around **0.28**), followed by **age** and **BMI** (around **0.15** and **0.14**). This matches clinical intuition: the model's predictions hinge mostly on blood glucose, with age and body mass as secondary signals.

## Visualising the Decision Surface in 3D

We can only plot in three dimensions, so we fix attention on the two most important features and average the predicted probability over the remaining ones.

In [ ]:
#We can visualise a maximum of 3 variables, so lets group by the 2 most important variables and compute the mean
grid_slice = full_grid.groupby(['glucose','bmi'],as_index=False).mean()

grid_slice

In [ ]:
#create a 3D visualisation of the resulting probability vs 2 most important prediction variables plot
fig = plt.figure()
ax = fig.add_subplot(projection='3d')

n = 100

xs = grid_slice.glucose
ys = grid_slice.bmi
zs = grid_slice.prob
ax.scatter3D(xs, ys, zs, marker='o', c = zs, cmap = 'rainbow')

ax.set_xlabel('glucose')
ax.set_ylabel('bmi')
ax.set_zlabel('probability')

plt.show()



The surface rises towards higher glucose and higher BMI: the model assigns a greater probability of diabetes in that corner of the feature space. Plotting the surface is a sanity check that the model behaves sensibly across the input range, not just a single accuracy number.

## Summary

In this notebook you:

- Trained an XGBoost classifier on the Pima diabetes dataset.
- Evaluated it with a classification report and a confusion matrix, reading per-class precision and recall on imbalanced data.
- Ranked the feature importances and found glucose, age, and BMI to be the strongest predictors.
- Mapped the decision surface over a feature grid and visualised it in 3D.

## References & Further Reading

- [**XGBoost Documentation**](https://xgboost.readthedocs.io/en/stable/): Official documentation for the gradient-boosting library.
- [**How to Develop Your First XGBoost Model in Python**](https://machinelearningmastery.com/develop-first-xgboost-model-python-scikit-learn/): A step-by-step walkthrough of training XGBoost with scikit-learn.
- [**Scikit-learn: Metrics and Scoring**](https://scikit-learn.org/stable/modules/model_evaluation.html): How accuracy, precision, recall, and F-score are defined and when to use each.